# ProjectPulse — Predictive Project Delay & Risk Analysis
**Track. Analyze. Predict. Deliver.**

### Data Science & Machine Learning Pipeline
This notebook demonstrates the end-to-end Data Science workflow for **ProjectPulse**, an AI-powered project risk forecasting platform.

> **Disclaimer:** The dataset used in this analysis is realistically simulated for the ProjectPulse internship prototype to demonstrate agile engineering dynamics and ML pipeline best practices without exposing proprietary organizational telemetry.

---

## 1. Environment Setup & Library Imports
Import core scientific and machine learning libraries.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Set plotting aesthetics
sns.set_theme(style="whitegrid", palette="deep")
%matplotlib inline

# Configure local source paths
current_dir = os.getcwd()
src_path = os.path.abspath(os.path.join(current_dir, "..", "src"))
if src_path not in sys.path:
    sys.path.append(src_path)

from data_cleaning import clean_project_data
from feature_engineering import compute_derived_features, prepare_features, FEATURE_COLUMNS, TARGET_COLUMN
from predict import predict_project_risk
print("All modules loaded successfully.")

## 2. Ingesting Raw Project Telemetry
Load the snapshot dataset from `dataset/project_data.csv`.

In [ ]:
data_csv = os.path.abspath(os.path.join(current_dir, "..", "dataset", "project_data.csv"))
raw_df = pd.read_csv(data_csv)
print(f"Raw dataset dimensions: {raw_df.shape}")
raw_df.head()

## 3. Data Cleaning & Sanitization
Execute sanitization pipeline: deduplication, missing value imputation, non-negativity enforcement, and task invariant reconciliation.

In [ ]:
cleaned_df, cleaning_summary = clean_project_data(raw_df)
print("Data Cleaning Operations Performed:")
for key, val in cleaning_summary.items():
    print(f"  - {key}: {val}")

print(f"Cleaned dataset dimensions: {cleaned_df.shape}")
print(f"Remaining missing values across columns: {cleaned_df.isnull().sum().sum()}")

## 4. Exploratory Data Analysis (EDA)
Analyze key operational distributions, class balance, and schedule risk drivers.

In [ ]:
# 4.1 Target Variable Distribution
plt.figure(figsize=(6, 4))
counts = cleaned_df['delay_status'].value_counts().sort_index()
labels = ['On Time (0)', 'Delayed (1)']
colors = ['#2563eb', '#e11d48']
bars = plt.bar(labels, counts, color=colors, width=0.45)
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h + 5, f"{h} ({h/len(cleaned_df):.1%})", ha='center', fontweight='bold')
plt.title("Project Delivery Status Distribution", fontweight='bold')
plt.ylabel("Count")
plt.ylim(0, max(counts)*1.15)
plt.show()

In [ ]:
# 4.2 Completion Percentage vs Delivery Status
comp_pct = (cleaned_df['completed_tasks'] / np.maximum(cleaned_df['total_tasks'], 1)) * 100
eda_df = cleaned_df.copy()
eda_df['completion_pct'] = comp_pct
eda_df['Status'] = eda_df['delay_status'].map({0: 'On Time', 1: 'Delayed'})

plt.figure(figsize=(7, 4.5))
sns.boxplot(x='Status', y='completion_pct', data=eda_df, hue='Status', palette={'On Time': '#2563eb', 'Delayed': '#e11d48'}, width=0.4, legend=False)
plt.title("Task Completion % by Delivery Status", fontweight='bold')
plt.ylabel("Completion Percentage (%)")
plt.show()

In [ ]:
# 4.3 Average Workload Density vs Delay Status
plt.figure(figsize=(7, 4.5))
sns.kdeplot(data=eda_df, x='avg_workload', hue='Status', palette={'On Time': '#2563eb', 'Delayed': '#e11d48'}, fill=True, common_norm=False, alpha=0.35)
plt.axvline(40.0, color='#d97706', linestyle='--', linewidth=1.5, label='Capacity (40h/wk)')
plt.title("Developer Workload Density vs Delay Status", fontweight='bold')
plt.xlabel("Average Workload (Hours/Week/Dev)")
plt.legend()
plt.show()

## 5. Feature Engineering (Leakage-Free)
Compute domain-engineered predictors:
- `completion_percentage`: $(completed\_tasks / total\_tasks) \times 100$
- `budget_utilization`: $(spent\_budget / budget) \times 100$
- `workload_ratio`: $avg\_workload / 40.0$
- `hour_variance`: $actual\_hours - estimated\_hours$
- `deadline_pressure`: $pending\_tasks / deadline\_days\_remaining$
- `dependency_risk`: $dependencies / total\_tasks$

*Target label `delay_status` is strictly isolated from predictors.*

In [ ]:
X, y = prepare_features(cleaned_df, is_training=True)
print(f"Engineered Predictors: {X.shape[1]} features")
print(f"Target vector length: {len(y)}")
X.describe().T[['mean', 'std', 'min', 'max']]

## 6. Model Training — Random Forest Classifier
Partition into 80% train / 20% holdout test with stratification and fit 100-tree ensemble.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set: {len(X_train)} samples | Test set: {len(X_test)} samples")

rf = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_depth=6,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    bootstrap=True,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print("Random Forest Classifier trained successfully.")

## 7. Model Evaluation & Benchmark Metrics
Evaluate against holdout test samples and inspect performance.

In [ ]:
y_pred = rf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("=" * 40)
print(f"Holdout Test Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Holdout Test Precision: {prec:.4f}")
print(f"Holdout Test Recall:    {rec:.4f}")
print(f"Holdout Test F1-Score:  {f1:.4f}")
print("=" * 40)
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['On Time', 'Delayed']))

In [ ]:
# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['On Time', 'Delayed'], yticklabels=['On Time', 'Delayed'])
plt.title("Confusion Matrix — Test Set", fontweight='bold')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()

## 8. Feature Importance Ranking
Extract Mean Decrease in Impurity (MDI) to identify key predictors.

In [ ]:
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
sorted_features = [FEATURE_COLUMNS[i] for i in indices]
sorted_weights = importances[indices]

plt.figure(figsize=(9, 5.5))
colors = plt.cm.Blues(np.linspace(0.85, 0.40, len(sorted_features)))
plt.barh(range(len(sorted_features)), sorted_weights[::-1], color=colors[::-1])
plt.yticks(range(len(sorted_features)), sorted_features[::-1])
plt.xlabel("Gini Importance")
plt.title("Feature Importance Ranking (Random Forest)", fontweight='bold')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 9. Real-Time Inference & Scenario Simulation
Validate the reusable `predict_project_risk()` function across canonical operational project profiles.

In [ ]:
scenarios = {
    "Low-Risk Project": {
        "team_size": 7,
        "total_tasks": 40,
        "completed_tasks": 36,
        "pending_tasks": 4,
        "avg_workload": 26.0,
        "estimated_hours": 850.0,
        "actual_hours": 700.0,
        "deadline_days_remaining": 45,
        "previous_delays": 0,
        "budget": 1400000.0,
        "spent_budget": 1050000.0,
        "task_complexity": 1.0,
        "dependencies": 1
    },
    "Medium-Risk Project": {
        "team_size": 5,
        "total_tasks": 50,
        "completed_tasks": 25,
        "pending_tasks": 25,
        "avg_workload": 39.5,
        "estimated_hours": 1200.0,
        "actual_hours": 620.0,
        "deadline_days_remaining": 28,
        "previous_delays": 1,
        "budget": 1800000.0,
        "spent_budget": 1000000.0,
        "task_complexity": 2.0,
        "dependencies": 5
    },
    "High-Risk Project": {
        "team_size": 3,
        "total_tasks": 55,
        "completed_tasks": 10,
        "pending_tasks": 45,
        "avg_workload": 58.0,
        "estimated_hours": 1400.0,
        "actual_hours": 1150.0,
        "deadline_days_remaining": 7,
        "previous_delays": 4,
        "budget": 1500000.0,
        "spent_budget": 1480000.0,
        "task_complexity": 3.0,
        "dependencies": 16
    }
}

for name, data in scenarios.items():
    res = predict_project_risk(data)
    print(f"=== {name} ===")
    print(f"  Delay Probability:   {res['delay_probability']:.2f}")
    print(f"  Composite Risk Score:{res['risk_score']} / 100")
    print(f"  Assigned Risk Tier:  {res['risk_level']}")
    print(f"  Expected Slip:       {res['expected_delay_days']} days\n")